# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faheem-danish/internship-starter-1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [30]:
from google.colab import userdata

token = userdata.get("HF_TOKEN")

print("Token loaded:", token is not None)

Token loaded: True


In [31]:
from huggingface_hub import hf_hub_download
import pandas as pd

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=token
)

print("March file:", march_file)

March file: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [32]:
march_df = pd.read_parquet(march_file)

print("Shape:", march_df.shape)
print(
    "Date range:",
    march_df["report_date"].min(),
    "to",
    march_df["report_date"].max()
)

Shape: (9841378, 30)
Date range: 2026-03-01 to 2026-03-31


In [33]:
baseline_df = march_df[
    march_df["gsc_data_available"].eq(True)
].copy()

baseline_df = baseline_df[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position"
    ]
].copy()

baseline_df = baseline_df.rename(
    columns={
        "gsc_impressions": "impressions",
        "gsc_clicks": "clicks",
        "gsc_avg_position": "avg_position"
    }
)

baseline_df["ctr_pct"] = (
    baseline_df["clicks"] /
    baseline_df["impressions"].replace(0, pd.NA)
) * 100

baseline_df.head()

,client_hash_id,content_hash_id,impressions,clicks,avg_position,ctr_pct
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,0.0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,0.0
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,0.8
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,0.0
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,0.0


In [34]:
print("Number of content items:", len(baseline_df))
print("Total impressions:", baseline_df["impressions"].sum())
print("Total clicks:", baseline_df["clicks"].sum())

print("\nMissing values:")
print(
    baseline_df[
        ["impressions", "clicks", "avg_position", "ctr_pct"]
    ].isna().sum()
)

Number of content items: 3611061
Total impressions: 280657589
Total clicks: 821832

Missing values:
impressions     0
clicks          0
avg_position    0
ctr_pct         0
dtype: int64


In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
volume_check = baseline_df.copy()

volume_check["volume_bucket"] = pd.cut(
    volume_check["impressions"],
    bins=[-1, 0, 99, 499, 999, 4999, float("inf")],
    labels=[
        "0",
        "1-99",
        "100-499",
        "500-999",
        "1000-4999",
        "5000+"
    ]
)

volume_table = (
    volume_check
    .groupby("volume_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        mean_impressions=("impressions", "mean"),
        mean_ctr_pct=("ctr_pct", "mean")
    )
    .reset_index()
)

volume_table

,volume_bucket,n,mean_impressions,mean_ctr_pct
0,0,0,NaN,NaN
1,1-99,2972453,20.242038,0.308647
2,100-499,537157,213.167126,0.310139
3,500-999,69032,682.367829,0.284563
4,1000-4999,31676,1659.839058,0.269738
5,5000+,743,8482.625841,0.344343


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
## My Rule

I will prioritize content that has meaningful search visibility but is not performing strongly for its current search position. The rule will focus on pages with at least 500 impressions and an average position of 11 or worse. Higher impressions increase the potential opportunity, while weaker positions indicate that the page may have room for improvement.

### Reason code

- **visible_position_opportunity** — the content has meaningful search visibility and an average position of 11 or worse, so it may be worth a refresh review.

### Action

The recommended action is **refresh_review**.

This is a transparent rule-based baseline, not a prediction of future performance.

In [36]:
position_check = baseline_df.copy()

position_check["position_bucket"] = pd.cut(
    position_check["avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=[
        "top_3",
        "4-10",
        "11-20",
        "21+"
    ],
    include_lowest=True
)

position_table = (
    position_check
    .groupby("position_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        mean_ctr_pct=("ctr_pct", "mean"),
        median_ctr_pct=("ctr_pct", "median")
    )
    .reset_index()
)

position_table

,position_bucket,n,mean_ctr_pct,median_ctr_pct
0,top_3,727362,0.475552,0.0
1,4-10,1456122,0.347264,0.0
2,11-20,519223,0.276991,0.0
3,21+,908354,0.128915,0.0


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [37]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Work from the content-level dataframe
df = content_df.copy()

# Eligibility: enough search visibility + position opportunity
df["eligible"] = (
    (df["impressions"] >= 500) &
    (df["avg_position"] >= 11)
)

# Transparent baseline score
# More impressions = larger opportunity.
# Worse position = larger opportunity.
position_factor = np.where(
    df["avg_position"] >= 21,
    1.5,
    1.0
)

df["score"] = (
    df["impressions"] *
    position_factor
)

# One reason code
df["reason_code"] = np.where(
    df["eligible"],
    "visible_position_opportunity",
    "not_eligible"
)

# Action label
df["action"] = np.where(
    df["eligible"],
    "refresh_review",
    "no_action"
)

# Rank eligible content
queue = (
    df[df["eligible"]]
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)

queue["rank"] = np.arange(1, len(queue) + 1)

print("Total content items:", len(df))
print("Selected for refresh review:", len(queue))
print("Top score:", round(queue["score"].max(), 2))

display(queue.head(20))

Total content items: 331437
Selected for refresh review: 21143
Top score: 291868.5


,client_hash_id,content_hash_id,impressions,clicks,avg_position,ctr_pct,eligible,score,reason_code,action,rank
0,client_23a62021009f63c4,content_36e53e9c707674fc,194579,242,32.766674,0.124371,True,291868.5,visible_position_opportunity,refresh_review,1
1,client_23a62021009f63c4,content_e8a52cf3d5988c07,244931,669,15.008339,0.273138,True,244931.0,visible_position_opportunity,refresh_review,2
2,client_20259bd6705d81d4,content_82e35c4845e6c391,143907,60,22.558608,0.041694,True,215860.5,visible_position_opportunity,refresh_review,3
3,client_23a62021009f63c4,content_3df3f32f3fd58dea,140156,197,23.335465,0.140558,True,210234.0,visible_position_opportunity,refresh_review,4
4,client_23a62021009f63c4,content_df47d1b976106de4,131707,163,24.355625,0.123760,True,197560.5,visible_position_opportunity,refresh_review,5
5,client_23a62021009f63c4,content_bdf60c86117079be,112429,12,30.769353,0.010673,True,168643.5,visible_position_opportunity,refresh_review,6
6,client_23a62021009f63c4,content_661a7734f691bef5,110424,73,23.888656,0.066109,True,165636.0,visible_position_opportunity,refresh_review,7
7,client_23a62021009f63c4,content_cae701a83cad5e36,98572,242,23.730705,0.245506,True,147858.0,visible_position_opportunity,refresh_review,8
8,client_23a62021009f63c4,content_559cdd76da9306de,97378,2,36.712074,0.002054,True,146067.0,visible_position_opportunity,refresh_review,9
9,client_20259bd6705d81d4,content_9fff53e827550f9d,94673,475,22.469914,0.501727,True,142009.5,visible_position_opportunity,refresh_review,10


In [38]:
import os

os.makedirs("work/outputs", exist_ok=True)

output_columns = [
    "client_hash_id",
    "content_hash_id",
    "impressions",
    "clicks",
    "avg_position",
    "ctr_pct",
    "eligible",
    "score",
    "reason_code",
    "action",
    "rank"
]

queue[output_columns].to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved:", "work/outputs/baseline_action_score.csv")

Saved: work/outputs/baseline_action_score.csv


In [39]:
print("Queue rows:", len(queue))
print(
    "Duplicate content items in queue:",
    queue.duplicated(
        ["client_hash_id", "content_hash_id"]
    ).sum()
)

Queue rows: 21143
Duplicate content items in queue: 0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*
## Top-20 Review

The top 20 items are ranked by a transparent opportunity score based on search impressions and average position.

For each item:

- **Action:** refresh_review
- **Reason:** visible_position_opportunity
- **Confidence:** Moderate, because the ranking is based on observed March search performance rather than a measured future outcome.
- **What could make it wrong:** The page may already be intentionally targeting a lower position, the query mix may not be relevant to the business, or the page may have constraints that make a refresh inappropriate.

The ranking is therefore a decision-support queue for human review, not an automatic instruction to change the content.

In [43]:
top20 = queue.head(20).copy()

top20["confidence_note"] = (
    "Moderate: based on observed March search performance; "
    "human review is required."
)

top20["what_would_make_it_wrong"] = (
    "Search intent may not match the page, or the page may have "
    "business/content constraints that make a refresh inappropriate."
)

display(
    top20[
        [
            "rank",
            "content_hash_id",
            "action",
            "reason_code",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ]
)

,rank,content_hash_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_36e53e9c707674fc,refresh_review,visible_position_opportunity,Moderate: based on observed March search perfo...,"Search intent may not match the page, or the p..."
1,2,content_e8a52cf3d5988c07,refresh_review,visible_position_opportunity,Moderate: based on observed March search perfo...,"Search intent may not match the page, or the p..."
2,3,content_82e35c4845e6c391,refresh_review,visible_position_opportunity,Moderate: based on observed March search perfo...,"Search intent may not match the page, or the p..."
3,4,content_3df3f32f3fd58dea,refresh_review,visible_position_opportunity,Moderate: based on observed March search perfo...,"Search intent may not match the page, or the p..."
4,5,content_df47d1b976106de4,refresh_review,visible_position_opportunity,Moderate: based on observed March search perfo...,"Search intent may not match the page, or the p..."
5,6,content_bdf60c86117079be,refresh_review,visible_position_opportunity,Moderate: based on observed March search perfo...,"Search intent may not match the page, or the p..."
6,7,content_661a7734f691bef5,refresh_review,visible_position_opportunity,Moderate: based on observed March search perfo...,"Search intent may not match the page, or the p..."
7,8,content_cae701a83cad5e36,refresh_review,visible_position_opportunity,Moderate: based on observed March search perfo...,"Search intent may not match the page, or the p..."
8,9,content_559cdd76da9306de,refresh_review,visible_position_opportunity,Moderate: based on observed March search perfo...,"Search intent may not match the page, or the p..."
9,10,content_9fff53e827550f9d,refresh_review,visible_position_opportunity,Moderate: based on observed March search perfo...,"Search intent may not match the page, or the p..."


In [40]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20).copy()

top20[
    [
        "rank",
        "content_hash_id",
        "impressions",
        "avg_position",
        "score",
        "reason_code",
        "action"
    ]
]

,rank,content_hash_id,impressions,avg_position,score,reason_code,action
0,1,content_36e53e9c707674fc,194579,32.766674,291868.5,visible_position_opportunity,refresh_review
1,2,content_e8a52cf3d5988c07,244931,15.008339,244931.0,visible_position_opportunity,refresh_review
2,3,content_82e35c4845e6c391,143907,22.558608,215860.5,visible_position_opportunity,refresh_review
3,4,content_3df3f32f3fd58dea,140156,23.335465,210234.0,visible_position_opportunity,refresh_review
4,5,content_df47d1b976106de4,131707,24.355625,197560.5,visible_position_opportunity,refresh_review
5,6,content_bdf60c86117079be,112429,30.769353,168643.5,visible_position_opportunity,refresh_review
6,7,content_661a7734f691bef5,110424,23.888656,165636.0,visible_position_opportunity,refresh_review
7,8,content_cae701a83cad5e36,98572,23.730705,147858.0,visible_position_opportunity,refresh_review
8,9,content_559cdd76da9306de,97378,36.712074,146067.0,visible_position_opportunity,refresh_review
9,10,content_9fff53e827550f9d,94673,22.469914,142009.5,visible_position_opportunity,refresh_review


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
## Weak Picks and Leakage Check

Some high-scoring pages may still be weak recommendations. For example, a page can have many impressions and a position below 10 because it targets broad or competitive queries where a refresh may not be the best action.

The baseline uses only March search-performance information available in the selected slice. It does not use future performance, product-decision flags, `trend_direction`, `trend_pct`, or other label-derived fields.

Therefore, the score should be treated as a directional decision-support baseline rather than proof that a page needs a refresh.

In [41]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage check
forbidden_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

used_columns = set(df.columns)

print("Leakage columns present in dataframe:")
print([c for c in forbidden_columns if c in used_columns])

print("\nFeatures used by baseline:")
print([
    "impressions",
    "avg_position"
])

print("\nFuture/label-derived fields are not used in the score.")

Leakage columns present in dataframe:
[]

Features used by baseline:
['impressions', 'avg_position']

Future/label-derived fields are not used in the score.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.